# 570. Managers with at Least 5 Direct Reports

[LeetCode problem](https://leetcode.com/problems/managers-with-at-least-5-direct-reports/)


# 0. Problem

Return manager names with at least five direct reports.


# 1. Setup


In [ ]:
import pandas as pd
employee_rows=[(101,'John','A',None),(102,'Dan','A',101),(103,'James','A',101),(104,'Amy','A',101),(105,'Anne','A',101),(106,'Ron','B',101),(107,'Eve','B',102)]
employee_pd=pd.DataFrame(employee_rows,columns=['id','name','department','managerId'])


In [ ]:
# In Colab if needed: !pip -q install pyspark
from pyspark.sql import SparkSession, functions as F, types as T
spark=SparkSession.builder.getOrCreate()
schema=T.StructType([T.StructField('id',T.IntegerType(),False),T.StructField('name',T.StringType(),False),T.StructField('department',T.StringType(),False),T.StructField('managerId',T.IntegerType(),True)])
employee_spark=spark.createDataFrame(employee_rows,schema)
employee_spark.createOrReplaceTempView('Employee')


# 2. SQL Solution


In [ ]:
sql_result=spark.sql("""SELECT m.name FROM Employee e JOIN Employee m ON e.managerId=m.id GROUP BY m.id,m.name HAVING COUNT(*)>=5""")
sql_result.show(truncate=False)


# 3. pandas Solution


In [ ]:
report_counts=employee_pd.dropna(subset=['managerId']).groupby('managerId',as_index=False).size().rename(columns={'size':'direct_reports'})
pandas_result=report_counts.loc[report_counts['direct_reports']>=5].merge(employee_pd[['id','name']],left_on='managerId',right_on='id',how='inner')[['name']]
pandas_result


# 4. PySpark Solution


In [ ]:
report_counts=employee_spark.filter(F.col('managerId').isNotNull()).groupBy('managerId').agg(F.count('*').alias('direct_reports')).filter(F.col('direct_reports')>=5)
spark_result=report_counts.join(employee_spark.select(F.col('id').alias('managerId'),'name'),on='managerId',how='inner').select('name')
spark_result.show(truncate=False)


# 5. Pattern Mapping

| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| group count | `COUNT(*)` | `.groupby().size()` | `F.count('*')` |
| group filter | `HAVING` | filter grouped DataFrame | `.filter()` after `.groupBy()` |
| manager lookup | self `JOIN` | `.merge()` | `.join()` |


# 6. Muscle-Memory Round


In [ ]:
# MUSCLE MEMORY — SQL
# Use temp view: Employee


In [ ]:
# MUSCLE MEMORY — PANDAS
# Use DataFrame: employee_pd


In [ ]:
# MUSCLE MEMORY — PYSPARK
# Use DataFrame: employee_spark
